# Event Category Clustering

Embed every event's text with a sentence-embedding model, cluster with K-means, and see how the
events *naturally* group — compared against the current hand-made categories. The goal is raw
material for inventing better category names.

Outputs land in `output/`: `cluster_summary.md` (share with Claude for naming), `assignments.csv`,
and `cluster_map.html`.

In [1]:
from pathlib import Path

TEXT_SOURCE = "both"  # "both" | "name" | "description"
MODEL_NAME = "all-mpnet-base-v2"
K = 6
K_SWEEP = range(4, 17)
RANDOM_STATE = 42

EVENTS_DIR = Path("../../public/events").resolve()
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load events

Categories and files come from `manifest.json`, so this picks up new categories automatically
(and skips non-manifest files like `candidates.json`).

In [2]:
import json

import pandas as pd

manifest = json.loads((EVENTS_DIR / "manifest.json").read_text())
rows = []
for cat in manifest["categories"]:
    for fname in cat["files"]:
        for ev in json.loads((EVENTS_DIR / fname).read_text()):
            rows.append(
                {
                    "name": ev["name"],
                    "friendly_name": ev["friendly_name"],
                    "description": ev["description"],
                    "category": cat["name"],
                    "year": ev["year"],
                }
            )
df = pd.DataFrame(rows)
print(f"{len(df)} events across {df['category'].nunique()} categories")
df["category"].value_counts()

4602 events across 7 categories


category
exploration       1176
diplomatic        1132
cultural           701
conflict           601
infrastructure     464
people             313
disasters          215
Name: count, dtype: int64

## 2. Build the text to embed

In [3]:
def build_text(row):
    if TEXT_SOURCE == "name":
        return row["friendly_name"]
    if TEXT_SOURCE == "description":
        return row["description"]
    return f"{row['friendly_name']}. {row['description']}"


df["text"] = df.apply(build_text, axis=1)
df["text"].head(3).tolist()

['Battle of Megiddo. The first battle recorded in detail, Pharaoh Thutmose III defeated a Canaanite coalition.',
 'Battle of Kadesh. Egypt and the Hittite Empire clashed in one of the largest chariot battles in history.',
 'Persian Wars Begin. Greek city-states revolted against Persian rule, beginning decades of conflict.']

## 3. Embed

First run downloads the model (~420 MB) and takes a minute; embeddings are cached in `output/`
keyed by a hash of the texts, so re-runs skip straight to clustering.

In [4]:
import hashlib

import numpy as np

cache_key = hashlib.md5((MODEL_NAME + "\n".join(df["text"])).encode()).hexdigest()[:12]
cache_file = OUTPUT_DIR / f"embeddings_{MODEL_NAME}_{TEXT_SOURCE}_{cache_key}.npy"

if cache_file.exists():
    embeddings = np.load(cache_file)
    print(f"Loaded cached embeddings from {cache_file}")
else:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(df["text"].tolist(), show_progress_bar=True, normalize_embeddings=True)
    np.save(cache_file, embeddings)
    print(f"Embedded and cached to {cache_file}")

embeddings.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/144 [00:00<?, ?it/s]

Embedded and cached to output/embeddings_all-mpnet-base-v2_both_da1603611938.npy


(4602, 768)

## 4. How many clusters does the data want?

Three lenses on each k:

- **Silhouette** (cosine): how cleanly separated the clusters are. Higher is better.
- **Stability**: mean pairwise ARI between K-means runs with different random seeds.
  If the same structure reappears regardless of seed, it's real; if runs disagree,
  the "clusters" are arbitrary slices. Higher is better.
- **Davies-Bouldin**: average similarity of each cluster to its nearest neighbor cluster.
  Lower is better.

The most *natural* k is where silhouette and stability are both high — a well-separated
structure that K-means finds reliably.

In [5]:
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score

N_STABILITY_RUNS = 5

sweep = []
for k in K_SWEEP:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(embeddings)
    sil = silhouette_score(embeddings, km.labels_, metric="cosine")
    db = davies_bouldin_score(embeddings, km.labels_)
    seed_labels = [
        KMeans(n_clusters=k, n_init=3, random_state=seed).fit_predict(embeddings)
        for seed in range(N_STABILITY_RUNS)
    ]
    stability = float(
        np.mean(
            [
                adjusted_rand_score(seed_labels[i], seed_labels[j])
                for i in range(N_STABILITY_RUNS)
                for j in range(i + 1, N_STABILITY_RUNS)
            ]
        )
    )
    sweep.append({"k": k, "silhouette": sil, "stability": stability, "davies_bouldin": db})
    print(f"k={k:2d}  silhouette={sil:.4f}  stability={stability:.3f}  davies_bouldin={db:.3f}")

sweep_df = pd.DataFrame(sweep)
best = sweep_df.loc[(sweep_df["silhouette"].rank() + sweep_df["stability"].rank()).idxmax(), "k"]
print(f"\nBest silhouette+stability combination: k={best}")

px.line(
    sweep_df.melt(id_vars="k", value_vars=["silhouette", "stability"], var_name="metric"),
    x="k",
    y="value",
    color="metric",
    markers=True,
    title="Cluster quality by k (both metrics: higher = better)",
)

k= 4  silhouette=0.0749  stability=0.730  davies_bouldin=4.261


k= 5  silhouette=0.0729  stability=0.699  davies_bouldin=4.368


k= 6  silhouette=0.0710  stability=0.781  davies_bouldin=4.016


k= 7  silhouette=0.0693  stability=0.633  davies_bouldin=4.155


k= 8  silhouette=0.0644  stability=0.700  davies_bouldin=4.085


...

k=12  silhouette=0.0732  stability=0.670  davies_bouldin=4.019


k=13  silhouette=0.0762  stability=0.664  davies_bouldin=3.957


k=14  silhouette=0.0782  stability=0.750  davies_bouldin=3.949


k=15  silhouette=0.0773  stability=0.760  davies_bouldin=3.871


k=16  silhouette=0.0785  stability=0.659  davies_bouldin=3.900

Best silhouette+stability combination: k=14


## 5. Cluster at k = K

In [6]:
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit(embeddings)
df["cluster"] = kmeans.labels_

centroids = kmeans.cluster_centers_ / np.linalg.norm(kmeans.cluster_centers_, axis=1, keepdims=True)
df["dist_to_centroid"] = 1 - np.einsum("ij,ij->i", embeddings, centroids[df["cluster"]])

df["cluster"].value_counts().sort_index()

cluster
0     970
1     343
2     695
3     459
4    1051
5    1084
Name: count, dtype: int64

## 6. What is each cluster about?

Top distinguishing terms (c-TF-IDF: each cluster's concatenated text is one document) and the
events closest to each centroid — the raw material for naming the cluster.

In [7]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

EXTRA_STOPWORDS = {
    "first", "world", "century", "began", "begins", "begun", "new", "known",
    "early", "later", "year", "years", "marking", "became", "history", "historic",
}
stop_words = list(ENGLISH_STOP_WORDS | EXTRA_STOPWORDS)

cluster_docs = [" ".join(df.loc[df["cluster"] == c, "text"]) for c in range(K)]
vectorizer = TfidfVectorizer(stop_words=stop_words, ngram_range=(1, 2), sublinear_tf=True)
ctfidf = vectorizer.fit_transform(cluster_docs)
vocab = np.array(vectorizer.get_feature_names_out())


def top_terms(c, n=12):
    weights = ctfidf[c].toarray().ravel()
    return vocab[weights.argsort()[::-1][:n]].tolist()


def fmt_year(y):
    return f"{-y:,} BCE" if y < 0 else str(y)


summary_lines = []
for c in range(K):
    sub = df[df["cluster"] == c]
    summary_lines.append(f"\n## Cluster {c} — {len(sub)} events")
    summary_lines.append(f"**Top terms:** {', '.join(top_terms(c))}")
    breakdown = sub["category"].value_counts()
    summary_lines.append(
        "**Came from:** " + ", ".join(f"{cat} ({n})" for cat, n in breakdown.items())
    )
    summary_lines.append("**Most central events:**")
    for ev in sub.nsmallest(15, "dist_to_centroid").itertuples():
        summary_lines.append(f"- {ev.friendly_name} ({fmt_year(ev.year)}) — {ev.description}")

print("\n".join(summary_lines))


## Cluster 0 — 970 events
**Top terms:** invented, patented, machine, developed, mechanical, turbine, carbon, demonstrated, clock, engine, battery, loom
**Came from:** exploration (751), infrastructure (84), diplomatic (64), cultural (61), conflict (10)
**Most central events:**
- Microscope Invented (1590) — Dutch spectacle makers invented the first compound microscope.
- Water Wheel Invented (350 BCE) — Ancient engineers developed the water wheel to harness river currents for grinding grain.
- Laser Invented (1960) — Theodore Maiman built the first working laser.
- Pottery Invented (14,000 BCE) — The earliest known ceramic vessels were created in Japan, marking the beginning of pottery.
- Zipper Invented (1893) — Whitcomb Judson patented the 'clasp-locker,' the precursor to the modern zipper.
- Gunpowder Invented (850) — Chinese alchemists accidentally discovered gunpowder while seeking immortality.
- Invention of the Sail (3,500 BCE) — Ancient Egyptians invented sails, enabling long

## 7. How do the clusters line up with the current categories?

ARI/NMI near 0 = no relation, near 1 = identical taxonomy. The heatmap shows which categories
survive intact and which get split or merged.

In [8]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

contingency = pd.crosstab(df["cluster"], df["category"])
ari = adjusted_rand_score(df["category"], df["cluster"])
nmi = normalized_mutual_info_score(df["category"], df["cluster"])
print(f"Adjusted Rand Index: {ari:.3f}")
print(f"Normalized Mutual Information: {nmi:.3f}")

px.imshow(
    contingency,
    text_auto=True,
    labels={"x": "current category", "y": "cluster", "color": "events"},
    title="Cluster vs. current category",
    color_continuous_scale="Blues",
)

Adjusted Rand Index: 0.274
Normalized Mutual Information: 0.369


## 8. Map

UMAP projection of the embeddings. Use the dropdown to switch between coloring by new cluster
and by current category — disagreements jump out. Also saved to `output/cluster_map.html`.

In [9]:
import plotly.graph_objects as go
import umap

reducer = umap.UMAP(metric="cosine", random_state=RANDOM_STATE)
coords = reducer.fit_transform(embeddings)
df["x"], df["y"] = coords[:, 0], coords[:, 1]


def scatter_traces(column, visible):
    traces = []
    for val in sorted(df[column].unique(), key=str):
        sub = df[df[column] == val]
        traces.append(
            go.Scattergl(
                x=sub["x"],
                y=sub["y"],
                mode="markers",
                name=str(val),
                visible=visible,
                marker={"size": 4, "opacity": 0.7},
                text=[
                    f"{ev.friendly_name}<br>{fmt_year(ev.year)} · {ev.category} · cluster {ev.cluster}"
                    for ev in sub.itertuples()
                ],
                hoverinfo="text",
            )
        )
    return traces


cluster_traces = scatter_traces("cluster", True)
category_traces = scatter_traces("category", False)
fig = go.Figure(cluster_traces + category_traces)
n_cl, n_cat = len(cluster_traces), len(category_traces)
fig.update_layout(
    title="Event map (UMAP of text embeddings)",
    height=700,
    updatemenus=[
        {
            "buttons": [
                {
                    "label": "Color by cluster",
                    "method": "update",
                    "args": [{"visible": [True] * n_cl + [False] * n_cat}],
                },
                {
                    "label": "Color by current category",
                    "method": "update",
                    "args": [{"visible": [False] * n_cl + [True] * n_cat}],
                },
            ],
            "x": 1.0,
            "y": 1.12,
        }
    ],
)
fig.write_html(OUTPUT_DIR / "cluster_map.html")
fig

/Users/emuir/Documents/GitHub/Vibes/timeline/when/experiments/category-clustering/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 9. Export

In [10]:
summary_md = "\n".join(
    [
        "# Cluster summary",
        f"Model: {MODEL_NAME} · text: {TEXT_SOURCE} · k={K} · {len(df)} events",
        f"ARI vs current categories: {ari:.3f} · NMI: {nmi:.3f}",
        "",
        "## Contingency (cluster × current category)",
        contingency.to_markdown(),
        *summary_lines,
    ]
)
(OUTPUT_DIR / "cluster_summary.md").write_text(summary_md)

df[["name", "friendly_name", "year", "category", "cluster", "dist_to_centroid"]].rename(
    columns={"category": "current_category"}
).to_csv(OUTPUT_DIR / "assignments.csv", index=False)

print("Wrote output/cluster_summary.md and output/assignments.csv")

Wrote output/cluster_summary.md and output/assignments.csv


## Next step

Share `output/cluster_summary.md` with Claude and ask: *"Here are the 6 clusters the data found —
what would you name each one as a game category?"* Try `TEXT_SOURCE = "description"` or a
different `K` and compare.